<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/data_analyse/data_analyse_d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#stores type annotations in a simpler form and helps avoid errors when referring to classes or data types that are defined later in the code.
from __future__ import annotations
#These are used to clearly describe the expected data types of function inputs and outputs.
from typing import Optional, Sequence, Tuple, Dict, Any, List
#pydantic is used to validate data before performing analysis.
from pydantic import BaseModel, ValidationError, field_validator

#work with tabular data such as CSV files, Excel sheets, or datasets.
import pandas as pd
#multi-format file loader
from pathlib import Path
import sqlite3
#numerical calculations, arrays, and mathematical operations.
import numpy as np
#built-in io module handles data stored temporarily in memory. It is useful when reading uploaded files without saving them permanently.
import io
#plotly.express is used to quickly create interactive charts with a small amount of code.
import plotly.express as px
#provides more control over chart design and customization than plotly.express.(For advanced chart design)
import plotly.graph_objects as go
#to place multiple charts inside one figure.
from plotly.subplots import make_subplots
#upload or download files when working in Google Colab.
from google.colab import files
#It provides statistical tests, probability distributions, optimization tools, and numerical methods.
import scipy
#chi2_contingency-checks whether two categorical variables are related , pointbiserialr-Used to measure the relationship between:one binary variable, such as 0 and 1,one numerical variable,such as age or salary
#f_oneway-compares the means of three or more groups, multivariate_normal- multivariate normal distribution
from scipy.stats import chi2_contingency, pointbiserialr, f_oneway,  multivariate_normal
#Converts categories into separate binary columns ,ordered categories into numerical values ,Scales numerical values into a fixed range, usually from 0 to 1 ,Scales data using the median and interquartile range
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler, RobustScaler
#to identify hidden underlying factors from several related variables
from sklearn.decomposition import FactorAnalysis




class DataAnalysisTool:
    def __init__(self):
        self.original_df = None
        self.df = None
        self.processed_df = None
        """
        constructor of your class.

        It runs automatically when you create an object from the class.
        ex:
        self.df = None
        tool = DataAnalysisTool()
        that moment, Python runs:__init__()  and creates the variables inside the object.
        create tool.df contains None. self refers to the current object.

        To restore the original data: self.original_df = None
        Encoded and scaled data for machine learning = self.processed_df = None
        """

    # -------------------------
    # 1. Load Data
    # -------------------------
    def upload_data(self):
        """
        Upload and load a tabular dataset in Google Colab.

        Supported formats:
        - CSV
        - Excel
        - JSON
        - TSV
        - TXT
        - Parquet

        Common missing-value symbols are converted into NaN.
        Mostly numeric columns are safely converted into numeric types.
        """

        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded.")
            return None

        # Select the first uploaded file
        file_name = next(iter(uploaded))

        # Get the uploaded file content
        file_content = uploaded[file_name]

        # Identify the file extension
        extension = Path(file_name).suffix.lower()

        missing_values = [
            "?",
            "n/a",
            "N/A",
            "NA",
            "NULL",
            "null",
            "None",
            "",
            " "
        ]

        # -------------------------
        # Load file based on format
        # -------------------------
        if extension == ".csv":
            self.df = pd.read_csv(
                io.BytesIO(file_content),
                na_values=missing_values
            )

        elif extension in [".xlsx", ".xls"]:
            self.df = pd.read_excel(
                io.BytesIO(file_content),
                na_values=missing_values
            )

        elif extension == ".json":
            self.df = pd.read_json(
                io.BytesIO(file_content)
            )

        elif extension == ".tsv":
            self.df = pd.read_csv(
                io.BytesIO(file_content),
                sep="\t",
                na_values=missing_values
            )

        elif extension == ".txt":
            self.df = pd.read_csv(
                io.BytesIO(file_content),
                sep=None,
                engine="python",
                na_values=missing_values
            )

        elif extension == ".parquet":
            self.df = pd.read_parquet(
                io.BytesIO(file_content)
            )

        else:
            raise ValueError(
                f"Unsupported file format: {extension}. "
                "Please upload CSV, Excel, JSON, TSV, TXT, or Parquet."
            )

        # -------------------------
        # Safely convert numeric columns
        # -------------------------
        for col in self.df.columns:
            numeric_col = pd.to_numeric(
                self.df[col],
                errors="coerce"
            )

            original_non_missing = self.df[col].notna().sum()

            if original_non_missing == 0:
                continue

            numeric_count = numeric_col.notna().sum()
            numeric_ratio = numeric_count / original_non_missing

            # Convert only if at least 80% of values are numeric
            if numeric_ratio >= 0.8:
                self.df[col] = numeric_col

        print(f"\n✅ File '{file_name}' loaded successfully!")
        print(f"File format: {extension}")
        print(f"Rows: {self.df.shape[0]}")
        print(f"Columns: {self.df.shape[1]}")

        return self.df

    # -------------------------
    # 2. Basic Summary
    # -------------------------
    def summary(self):
        if self.df is None:
            print("No data loaded.")
            return

        print("Rows:", self.df.shape[0])
        print("Columns:", self.df.shape[1])
        print("\nNumerical columns:")
        print(self.df.select_dtypes(include=np.number).columns.tolist())

        print("\nCategorical columns:")
        print(self.df.select_dtypes(exclude=np.number).columns.tolist())

        return self.df.head()

    # -------------------------
    # 3. Missing Data
    # -------------------------
    def missing_summary(self):
        if self.df is None:
            print("No data loaded.")
            return

        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        return missing

    def fill_missing(self, strategy="median", fill_value=None):
        if self.df is None:
            print("No data loaded.")
            return

        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0:
                continue

            if strategy == "mean" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].mean())

            elif strategy == "median" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].median())

            elif strategy == "mode":
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])

            elif strategy == "constant":
                self.df[col] = self.df[col].fillna(fill_value)

        print("Missing values handled.")
        return self.df

    # -------------------------
    # 4. Remove Duplicates
    # -------------------------
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        after = len(self.df)

        print(f"Removed {before - after} duplicate rows.")
        return self.df

    # -------------------------
    # 5. Outlier Detection
    # -------------------------
    def detect_outliers(self, columns=None):
        if columns is None:
            columns = self.df.select_dtypes(include=np.number).columns

        outlier_indices = set()

        for col in columns:
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            outliers = self.df[(self.df[col] < lower) | (self.df[col] > upper)]
            outlier_indices.update(outliers.index)

            print(f"{col}: {len(outliers)} outliers found")

        return self.df.loc[list(outlier_indices)]

    def remove_outliers(self, columns=None):
        outliers = self.detect_outliers(columns)
        self.df = self.df.drop(index=outliers.index).reset_index(drop=True)

        print("Outliers removed.")
        return self.df

    # -------------------------
    # 6. Encoding Categorical Data
    # -------------------------
    def encode_categorical(self, method="onehot"):
        cat_df = self.df.select_dtypes(exclude=np.number)

        if cat_df.empty:
            print("No categorical columns found.")
            return pd.DataFrame()

        if method == "onehot":
            encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            encoded = encoder.fit_transform(cat_df.fillna("Missing"))
            cols = encoder.get_feature_names_out(cat_df.columns)
            return pd.DataFrame(encoded, columns=cols)

        elif method == "ordinal":
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(cat_df.fillna("Missing"))
            return pd.DataFrame(encoded, columns=cat_df.columns)

    # -------------------------
    # 7. Scale Numerical Data
    # -------------------------
    def scale_numeric(self, method="standard"):
        num_df = self.df.select_dtypes(include=np.number)

        if method == "minmax":
            scaler = MinMaxScaler()
        elif method == "robust":
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        scaled = scaler.fit_transform(num_df.fillna(num_df.median()))
        return pd.DataFrame(scaled, columns=num_df.columns)

    # -------------------------
    # 8. Create ML Dataset
    # -------------------------
    def create_ml_dataset(self, scaling="standard", encoding="onehot"):
        num_scaled = self.scale_numeric(method=scaling)
        cat_encoded = self.encode_categorical(method=encoding)

        final_df = pd.concat([num_scaled, cat_encoded], axis=1)
        return final_df

    # -------------------------
    # 9. Numerical Correlation
    # -------------------------
    def numerical_correlation(self):
        num_df = self.df.select_dtypes(include=np.number)
        corr = num_df.corr()

        fig = px.imshow(
            corr,
            text_auto=True,
            title="Numerical Correlation Heatmap"
        )
        fig.show()

        return corr

    # -------------------------
    # 10. Categorical Correlation
    # -------------------------
    def categorical_correlation(self):
        cat_df = self.df.select_dtypes(exclude=np.number)
        cols = cat_df.columns

        matrix = pd.DataFrame(
            np.zeros((len(cols), len(cols))),
            index=cols,
            columns=cols
        )

        for col1 in cols:
            for col2 in cols:
                table = pd.crosstab(cat_df[col1], cat_df[col2])

                if table.shape[0] > 1 and table.shape[1] > 1:
                    chi2 = chi2_contingency(table)[0]
                    n = table.sum().sum()
                    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
                else:
                    v = 0

                matrix.loc[col1, col2] = v

        fig = px.imshow(
            matrix,
            text_auto=True,
            title="Categorical Association Heatmap"
        )
        fig.show()

        return matrix

    # -------------------------
    # 11. Plots
    # -------------------------
    def plot_histogram(self, column):
        fig = px.histogram(self.df, x=column, title=f"Histogram of {column}")
        fig.show()

    def plot_boxplot(self, column):
        fig = px.box(self.df, y=column, title=f"Boxplot of {column}")
        fig.show()

    def plot_scatter(self, x, y):
        fig = px.scatter(
            self.df,
            x=x,
            y=y,
            trendline="ols",
            title=f"{x} vs {y}"
        )
        fig.show()

    def plot_bar(self, column):
        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, "count"]

        fig = px.bar(
            counts,
            x=column,
            y="count",
            title=f"Bar Chart of {column}"
        )
        fig.show()

    # -------------------------
    # 12. Export
    # -------------------------
    def export_csv(self, filename="cleaned_data.csv"):
        self.df.to_csv(filename, index=False)
        print(f"Saved as {filename}")